# vitalPeriodic

"Periodic" data refers to data which is consistently interfaced from bedside vital signs monitors into eCareManager. Data are generally interfaced as 1 minute averages, and archived into the vitalPeriodic table as 5 minute median values.

The vital signs are **not** validated by care staff, so data quality can vary, but the use of 1 minute averages followed by 5 minute medians before archival removes many spurious readings.

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg2
import getpass
import pdvega

# for configuring connection 
from configobj import ConfigObj
import os
from loguru import logger

import holoviews as hv
from holoviews import opts
hv.extension("bokeh")
#hv.renderer('bokeh').webgl = True

from plot_helpers import layout_hv_curves, layout_hv_scatter

%matplotlib inline

In [ ]:
# Create a database connection using settings from config file
config='../db/config.ini'

# connection info
conn_info = dict()
if os.path.isfile(config):
    config = ConfigObj(config)
    conn_info["sqluser"] = config['username']
    conn_info["sqlpass"] = config['password']
    conn_info["sqlhost"] = config['host']
    conn_info["sqlport"] = config['port']
    conn_info["dbname"] = config['dbname']
    conn_info["schema_name"] = config['schema_name']
else:
    conn_info["sqluser"] = 'postgres'
    conn_info["sqlpass"] = ''
    conn_info["sqlhost"] = '172.25.93.63'
    conn_info["sqlport"] = 5432
    conn_info["dbname"] = 'eicu'
    conn_info["schema_name"] = 'public,eicu_crd'
    
# Connect to the eICU database
print('Database: {}'.format(conn_info['dbname']))
print('Username: {}'.format(conn_info["sqluser"]))
if conn_info["sqlpass"] == '':
    # try connecting without password, i.e. peer or OS authentication
    try:
        if (conn_info["sqlhost"] == 'localhost') & (conn_info["sqlport"]=='5432'):
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   user=conn_info["sqluser"])            
        else:
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   host=conn_info["sqlhost"],
                                   port=conn_info["sqlport"],
                                   user=conn_info["sqluser"])
    except:
        conn_info["sqlpass"] = getpass.getpass('Password: ')

        con = psycopg2.connect(dbname=conn_info["dbname"],
                               host=conn_info["sqlhost"],
                               port=conn_info["sqlport"],
                               user=conn_info["sqluser"],
                               password=conn_info["sqlpass"])
query_schema = 'set search_path to ' + conn_info['schema_name'] + ';'

In [ ]:
from sqlalchemy import create_engine

db_credentials_url = "".join(
    [config['username'], ":", config['password'], "@", config['host'], ":", config['port'], "/", config['dbname']])
database_url = "".join(["postgresql+psycopg2://", db_credentials_url])

logger.info(f"Connecting to the database with database_url={database_url}")

engine = create_engine(database_url)

## Examine patients 

In [ ]:
patientunitstayid = 145467

In [ ]:
# this query overflows the ram
#query = query_schema + """
#SELECT *
#FROM vitalperiodic
#ORDER BY patientunitstayid, observationoffset
#"""

In [ ]:
query_ids = query_schema + """
SELECT DISTINCT patientunitstayid
FROM vitalperiodic
ORDER BY patientunitstayid
"""

In [ ]:
df_unique_patientstays_periodic = pd.read_sql_query(query_ids, engine)
df_unique_patientstays_periodic

## Set the number of samples to visualize here

In [ ]:
count_stay_ids = 200
selected_ids = list(df_unique_patientstays_periodic.iloc[0:count_stay_ids,0].values)

In [ ]:

query_selected_ids = query_schema + """
SELECT *
FROM vitalperiodic
WHERE patientunitstayid IN ({})
ORDER BY patientunitstayid, observationoffset
""".format(', '.join(map(str, selected_ids)))

In [ ]:

df_periodic = pd.read_sql_query(query_selected_ids, engine)
df_periodic_offset = df_periodic.set_index('observationoffset', inplace=False)
df_periodic_offset_sorted = df_periodic_offset.sort_index(inplace=True)
#df_periodic = df_periodic_offset_sorted.reset_index()
#df_periodic.head()

In [ ]:
df_periodic.columns

In [ ]:
# list of columns to plot
vitals_1 = ['temperature', 'sao2', 'heartrate', 'respiration',
          'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'etco2']

In [ ]:
#df[vitals].vgplot.line()
layout_vital_1 = layout_hv_curves(dataframe_to_plot=df_periodic, columns_to_plot=vitals_1, title="Vital 1 Periodic TS")

layout_vital_1

As we can see most data is fairly consistent but there are a few clear outliers for respiration likely due to artifacts around the 2000 minute mark.

In [ ]:
# list of columns to plot
vitals_2 = ['cvp', 'pasystolic', 'padiastolic', 'pamean']
#df[vitals].vgplot.line()

In [ ]:
layout_vital_2 = layout_hv_curves(dataframe_to_plot=df_periodic, columns_to_plot=vitals_2, title="Vital 2 Periodic TS")


layout_vital_2

Though rare, this patient appears to have pulmonary artery monitoring which ceases at around the 1200 minute mark. This also coincides with a large increase in measured central venous pressure (CVP), which is probably related to the catheter removal.

In [ ]:
# list of columns to plot
vitals_3 = ['st1', 'st2', 'st3']
# no 'icp' data

layout_vital_3  = layout_hv_curves(dataframe_to_plot=df_periodic, columns_to_plot=vitals_3, title="Vital 3 Periodic TS")


layout_vital_3


In [ ]:
layout_periodic_all = hv.Layout([layout_vital_1 + layout_vital_2 + layout_vital_3]).cols(1)
layout_periodic_all.opts(
    shared_axes=False, normalize=False, title=f"Subset of 0th to {count_stay_ids}th stays")
layout_periodic_all

Machine derived ST segment levels are also available - quantization of the value is done at 0.1 mV and is readily observable.

Most hospitals in eICU have good data coverage for the vitalperiodic table.

In [ ]:
import panel as pn
from panel.template import DarkTheme

pn.extension(sizing_mode="stretch_width")
#pn.extension()
pn.extension()

panel_page = pn.template.BootstrapTemplate(title="Time Series Viewer", theme=DarkTheme, header_color="red")

layout_periodic_all = layout_periodic_all.map(lambda x: x.opts(responsive=True), hv.Element)

panel_page.main.append(pn.Row(layout_periodic_all, sizing_mode='stretch_width'))

panel_page.show()

In [ ]:
from bokeh.resources import INLINE

#panel_page.save(f'panel_exports/vital_periodic_subset_count_{count_stay_ids}.html', resources=INLINE)